# ♻️ Waste Intelligence - MobileNetV2 Model Training in Google Colab

This notebook trains and exports the **MobileNetV2** image classifier for the **Waste Intelligence** framework, following the exact specifications from the paper:

* **Architecture**: MobileNetV2 (pretrained on ImageNet, fine-tuned)
* **Input Shape**: `(224, 224, 3)`
* **Categories (C = 6)**: `["Paper", "Plastic", "Glass", "Metal", "Organic", "Other"]`
* **Optimizer**: Adam (learning rate = $1 \times 10^{-4}$)
* **Batch Size**: 32
* **Epochs**: 30 (with EarlyStopping & ModelCheckpoint)
* **Target Output**: `waste_classifier.keras` and `class_names.json`

> ⚡ **Tip**: Make sure your Colab runtime is set to **GPU** (`Runtime -> Change runtime type -> T4 GPU`).

## 1. Verify GPU & Environment

In [ ]:
import tensorflow as tf
import os, json, shutil
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print(f"TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ GPU Detected: {gpus[0].name}")
else:
    print("⚠️ No GPU detected. Running on CPU. Go to 'Runtime -> Change runtime type' and select 'T4 GPU' for faster training.")

## 2. Configuration & Hyperparameters (Table 4.1)

In [ ]:
# The exact 6 categories required by config.py
CLASS_NAMES = ["Paper", "Plastic", "Glass", "Metal", "Organic", "Other"]
NUM_CLASSES = len(CLASS_NAMES)
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
EPOCHS = 30
SEED = 42

DATA_DIR = Path("dataset")
for c in CLASS_NAMES:
    (DATA_DIR / c).mkdir(parents=True, exist_ok=True)

print("Classes:", CLASS_NAMES)

## 3. Dataset Setup

You can choose **Option A** (download the popular Garbage Classification dataset from Kaggle), **Option B** (use your own uploaded images from Google Drive), or **Option C** (generate a synthetic test dataset to verify the pipeline immediately).

In [ ]:
# Choose dataset source: 'kaggle', 'drive', or 'demo'
DATASET_SOURCE = 'kaggle'  # change to 'drive' or 'demo' if desired

if DATASET_SOURCE == 'kaggle':
    # Option A: Download from Kaggle (mostafaabla/garbage-classification or similar)
    # Note: Requires kaggle.json uploaded or KAGGLE_USERNAME / KAGGLE_KEY set.
    # If you don't have kaggle credentials, choose 'demo' below to verify the pipeline!
    try:
        import kagglehub
        print("Downloading dataset via kagglehub...")
        path = kagglehub.dataset_download("mostafaabla/garbage-classification")
        print("Downloaded to:", path)
        
        # Map Kaggle 12 classes to the project's 6 classes
        class_mapping = {
            "cardboard": "Paper", "paper": "Paper",
            "plastic": "Plastic",
            "brown-glass": "Glass", "green-glass": "Glass", "white-glass": "Glass",
            "metal": "Metal",
            "biological": "Organic",
            "battery": "Other", "clothes": "Other", "shoes": "Other", "trash": "Other"
        }
        
        source_root = Path(path)
        # Search for class subfolders
        subdirs = [d for d in source_root.rglob("*") if d.is_dir() and d.name.lower() in class_mapping]
        for sub in subdirs:
            target_class = class_mapping[sub.name.lower()]
            target_dir = DATA_DIR / target_class
            for img_file in list(sub.glob("*.*"))[:350]:  # balanced subset
                if img_file.suffix.lower() in [".jpg", ".jpeg", ".png"]:
                    shutil.copy(img_file, target_dir / f"{sub.name}_{img_file.name}")
                    
        print("Dataset prepared successfully from Kaggle!")
    except Exception as e:
        print(f"⚠️ Kaggle download failed or skipped ({e}). Falling back to 'demo' dataset generator...")
        DATASET_SOURCE = 'demo'

if DATASET_SOURCE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    print("Mounted Google Drive. Place your images in 'dataset/<Class>' or copy from drive.")

if DATASET_SOURCE == 'demo':
    print("Generating demo samples for all 6 classes to train a functional model...")
    from PIL import Image
    for c in CLASS_NAMES:
        folder = DATA_DIR / c
        for i in range(40):
            # Create pattern images for demonstration
            arr = np.random.randint(50, 220, (224, 224, 3), dtype=np.uint8)
            Image.fromarray(arr).save(folder / f"sample_{i}.jpg")

for c in CLASS_NAMES:
    count = len(list((DATA_DIR / c).glob("*.*")))
    print(f"  Class '{c}': {count} images")

## 4. Data Loading & Augmentation Pipeline

In [ ]:
# Load train and validation splits (80% train, 20% validation)
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    labels="inferred",
    label_mode="categorical",
    class_names=CLASS_NAMES,
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    shuffle=True,
    seed=SEED,
    validation_split=0.2,
    subset="training"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    labels="inferred",
    label_mode="categorical",
    class_names=CLASS_NAMES,
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    shuffle=False,
    seed=SEED,
    validation_split=0.2,
    subset="validation"
)

# Data augmentation for training robustness
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.1),
], name="data_augmentation")

# Prefetch for fast GPU execution
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
print("Classes in dataset order:", CLASS_NAMES)

## 5. Model Architecture: MobileNetV2 Transfer Learning

Per Table 4.1:
- MobileNetV2 backbone (ImageNet pre-trained)
- Scaling: `x / 127.5 - 1` (Standard MobileNetV2 preprocessing)
- GlobalAveragePooling2D + Dropout (0.2)
- Dense output layer with 6 units and **softmax** activation

In [ ]:
def create_model():
    inputs = tf.keras.Input(shape=(224, 224, 3), name="input_layer")
    x = data_augmentation(inputs)
    # MobileNetV2 standard normalisation (maps [0, 255] to [-1, 1])
    x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
    
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(224, 224, 3),
        include_top=False,
        weights="imagenet"
    )
    base_model.trainable = True
    # Freeze early feature extraction layers, fine-tune the top layers
    for layer in base_model.layers[:-30]:
        layer.trainable = False
        
    x = base_model(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D(name="gap")(x)
    x = tf.keras.layers.Dropout(0.2, name="dropout")(x)
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", name="predictions")(x)
    
    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="waste_classifier_mobilenetv2")
    return model

model = create_model()
model.summary()

## 6. Training with Adam Optimizer (LR = 1e-4, 30 Epochs)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=["accuracy", tf.keras.metrics.Precision(name="precision"), tf.keras.metrics.Recall(name="recall")]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
    tf.keras.callbacks.ModelCheckpoint("best_model.keras", monitor="val_accuracy", save_best_only=True)
]

print(f"Training MobileNetV2 for up to {EPOCHS} epochs...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

## 7. Model Evaluation

In [ ]:
val_loss, val_acc, val_prec, val_rec = model.evaluate(val_ds)
print("=" * 40)
print(f"Validation Accuracy : {val_acc * 100:.2f}%")
print(f"Validation Precision: {val_prec:.4f}")
print(f"Validation Recall   : {val_rec:.4f}")
print("=" * 40)

# Plot accuracy and loss curves
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

## 8. Export Model & Class Names for Waste Intelligence

The Waste Intelligence app expects exactly:
1. `models/waste_classifier.keras`
2. `models/class_names.json`

In [ ]:
# Export files
model_filename = "waste_classifier.keras"
class_filename = "class_names.json"

model.save(model_filename)
with open(class_filename, "w") as f:
    json.dump(CLASS_NAMES, f, indent=2)

print(f"✅ Saved model to: {model_filename} ({os.path.getsize(model_filename) / (1024*1024):.1f} MB)")
print(f"✅ Saved class names to: {class_filename}")
with open(class_filename) as f:
    print("Class names content:", f.read())

## 9. Download Files to Your Computer

Run this cell to download both files directly into your browser's download folder. Then move them into your local `waste-intelligence/models/` folder!

In [ ]:
from google.colab import files

print("Downloading waste_classifier.keras...")
files.download(model_filename)

print("Downloading class_names.json...")
files.download(class_filename)
print("✅ Done! Once downloaded, copy both files into your local 'waste-intelligence/models/' folder.")